# Stage 7 — Trajectory Divergence Analysis

**The novel diagnostic enabled by CTLS:**

Standard interpretability tools (GradCAM, LIME, probing) tell you *where in the image* or *what class* a model responds to. They cannot answer: **at which layer did this specific image's processing go wrong?**

CTLS makes this possible without any additional training. Because the backbone captures a full 8-layer activation trajectory — globally-pooled and soft-masked at each block — we can compute, for any image, its cosine similarity to each class centroid *at each layer independently*. This gives a depth-resolved picture of model behaviour at the individual-sample level.

**Four experiments:**

1. **Layer-collapse diagnostic** — per-layer within-class vs between-class distance. Validates and extends the Stage 4 finding: layer-7 collapse visible as a gap compression, not just an aggregate metric.
2. **Divergence profiles: correct vs misclassified** — average distance-to-true-centroid across depth. Misclassified images show systematically higher distance starting from an identifiable layer.
3. **Defection layer distribution** — for each misclassified image, the first layer where cosine similarity to the predicted (wrong) class exceeds similarity to the true class. Broken down by confusion pair.
4. **Per-image trajectory heatmaps** — 10-class × 8-layer cosine similarity heatmap for individual images. The visual payoff: watching the model commit to the wrong class in real time across depth.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from evaluation.circuit_analysis import CircuitAnalyzer, denormalize, CIFAR10_CLASSES
from data.cifar import get_standard_loaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

with open('configs/ctls.yaml') as f:
    config = yaml.safe_load(f)

mcfg = config['model']
ecfg = mcfg['meta_encoder']

soft_mask = SoftMask(init_temperature=0.1)

backbone = CTLSBackbone(
    arch=mcfg['arch'],
    num_classes=mcfg['num_classes'],
    soft_mask=soft_mask,
    pretrained=False,
).to(DEVICE)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load('experiments/ctls/best.pt', map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval()
meta_encoder.eval()

for p in backbone.parameters():     p.requires_grad_(False)
for p in meta_encoder.parameters(): p.requires_grad_(False)

print(f"Loaded: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=128,
    num_workers=dcfg.get('num_workers', 4),
    augment=False,
    download=True,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, DEVICE)
L = len(backbone.layer_dims)   # 8 for ResNet18
LAYER_LABELS = [f'L{i+1}' for i in range(L)]
print(f"Trajectory depth: {L} layers — dims {backbone.layer_dims}")

In [ ]:
print("Collecting raw per-layer trajectories (10 000 val samples)...")
trajs, logits_all, labels_all = analyzer.collect_trajectories(max_samples=10000)
print(f"  {len(labels_all)} samples collected")
print(f"  Layer dims: {[t.shape[1] for t in trajs]}")

print("Computing per-layer class centroids...")
layer_cents = analyzer.layer_class_centroids(trajs, labels_all)

preds_all = logits_all.argmax(dim=1)
correct_mask = preds_all == labels_all
print(f"  Val accuracy: {correct_mask.float().mean():.4f}")
print(f"  Misclassified: {(~correct_mask).sum().item()} images")

---

## 1. Layer-Collapse Diagnostic

At each layer, compute:
- **Within-class distance**: mean cosine distance between an image and its true-class centroid (lower = tighter cluster)
- **Between-class distance**: mean cosine distance between centroids of different class pairs (higher = better separated)

The **separation ratio** = between / within. A healthy layer has a high ratio. A collapsed layer shows both distances compressing toward the same value — the ratio drops.

Stage 4 reported layer-7 collapse in the aggregate trajectory metric. This plot localises it to an exact layer index and shows whether it affects all classes equally.

In [ ]:
within_dists = []   # [L]
between_dists = []  # [L]

for l in range(L):
    # Within-class: mean distance of each sample to its class centroid
    all_d = []
    for cls in range(10):
        mask = labels_all == cls
        h = F.normalize(trajs[l][mask], dim=-1)          # [M, D_l]
        cent = layer_cents[l][cls].unsqueeze(0)           # [1, D_l]
        d = 1 - (h * cent).sum(dim=-1)                   # [M]
        all_d.append(d)
    within_dists.append(torch.cat(all_d).mean().item())

    # Between-class: mean cosine distance between all pairs of centroids
    cents = torch.stack([layer_cents[l][c] for c in range(10)])  # [10, D_l]
    sim_mat = cents @ cents.T                                      # [10, 10]
    upper = sim_mat[torch.triu(torch.ones(10, 10), diagonal=1).bool()]
    between_dists.append((1 - upper).mean().item())

separation = [b / w if w > 0 else 0 for b, w in zip(between_dists, within_dists)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Stage 7 — Layer-Collapse Diagnostic', fontsize=13)

ax = axes[0]
ax.plot(LAYER_LABELS, within_dists,  'o-', color='steelblue',  label='Within-class dist')
ax.plot(LAYER_LABELS, between_dists, 's-', color='darkorange', label='Between-class dist')
ax.set_ylabel('Mean cosine distance')
ax.set_title('Within vs Between-class distance per layer')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
bars = ax.bar(LAYER_LABELS, separation, color=[
    'tomato' if s == min(separation) else 'steelblue' for s in separation
])
ax.set_ylabel('Separation ratio (between / within)')
ax.set_title('Class separation per layer  (red = collapse)')
ax.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, separation):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8)

fig.tight_layout()
os.makedirs('experiments/stage7', exist_ok=True)
fig.savefig('experiments/stage7/layer_collapse.png', dpi=150, bbox_inches='tight')
plt.show()

collapse_layer = LAYER_LABELS[separation.index(min(separation))]
print(f"Lowest separation at: {collapse_layer}")

---

## 2. Divergence Profiles: Correct vs Misclassified

For each image, compute the per-layer cosine distance to its true-class centroid.
Average separately over correctly classified and misclassified images.

**What to look for:**
- If misclassified images show *higher* distance from early layers → they were never well-represented; the model never had the right signal.
- If the gap opens *late* (layers 6–8) → the model had the right representation through most of the network but lost it at the collapse layer. This is the layer-7 collapse hypothesis expressed at the per-sample level.
- The layer where the correct/incorrect gap is largest is the most diagnostic layer in the network.

In [ ]:
print("Computing divergence curves (all val samples)...")

curves_correct = []
curves_wrong   = []

for i in range(len(labels_all)):
    traj_i  = [trajs[l][i] for l in range(L)]
    true_cls = labels_all[i].item()
    curve    = analyzer.trajectory_divergence_curve(traj_i, true_cls, layer_cents)
    if correct_mask[i]:
        curves_correct.append(curve)
    else:
        curves_wrong.append(curve)

curves_correct = torch.stack(curves_correct)  # [N_correct, L]
curves_wrong   = torch.stack(curves_wrong)    # [N_wrong, L]

mean_c = curves_correct.mean(0).numpy()
std_c  = curves_correct.std(0).numpy()
mean_w = curves_wrong.mean(0).numpy()
std_w  = curves_wrong.std(0).numpy()

x = np.arange(L)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, mean_c, 'o-', color='steelblue',  label=f'Correct ({len(curves_correct)})')
ax.fill_between(x, mean_c - std_c, mean_c + std_c, alpha=0.15, color='steelblue')
ax.plot(x, mean_w, 's-', color='tomato',     label=f'Misclassified ({len(curves_wrong)})')
ax.fill_between(x, mean_w - std_w, mean_w + std_w, alpha=0.15, color='tomato')
ax.set_xticks(x); ax.set_xticklabels(LAYER_LABELS)
ax.set_ylabel('Mean cosine distance to true-class centroid')
ax.set_title('Stage 7 — Divergence Profile: Correct vs Misclassified')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/stage7/divergence_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

gap = mean_w - mean_c
print("Gap (misclassified - correct) per layer:")
for i, (lbl, g) in enumerate(zip(LAYER_LABELS, gap)):
    print(f"  {lbl}: {g:+.4f}")
print(f"Largest gap at: {LAYER_LABELS[gap.argmax()]}")

---

## 3. Defection Layer Distribution

For each misclassified image, find the **defection layer**: the first layer at which cosine similarity to the *predicted (wrong)* class exceeds similarity to the *true* class.

This is the layer where the model's internal representation crosses over — where it stops processing the image like its true class and starts processing it like the predicted class.

**Three analyses:**
- Histogram of defection layers across all misclassified images
- Broken down by confusion pair (which pairs defect early vs late)
- Cross-reference with the collapse layer found in Section 1

In [ ]:
defections = []  # list of (true_cls, pred_cls, defection_layer_or_None)

misclassified_idx = torch.where(~correct_mask)[0]
print(f"Computing defection layers for {len(misclassified_idx)} misclassified images...")

for i in misclassified_idx.tolist():
    traj_i   = [trajs[l][i] for l in range(L)]
    true_cls = labels_all[i].item()
    pred_cls = preds_all[i].item()

    defect_l = None
    for l, h in enumerate(traj_i):
        sims = analyzer.layer_class_similarities(h, layer_cents[l])
        if sims[pred_cls] > sims[true_cls]:
            defect_l = l
            break

    defections.append((true_cls, pred_cls, defect_l))

# ---- global histogram ----
defect_layers = [d[2] for d in defections if d[2] is not None]
no_defect     = sum(1 for d in defections if d[2] is None)
print(f"  Defection found: {len(defect_layers)} / {len(defections)}")
print(f"  No defection (confusion throughout): {no_defect}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(defect_layers, bins=range(L + 1), align='left', rwidth=0.8,
        color='steelblue', edgecolor='white')
ax.set_xticks(range(L))
ax.set_xticklabels(LAYER_LABELS)
ax.set_xlabel('Defection layer (first layer where sim(pred) > sim(true))')
ax.set_ylabel('Number of misclassified images')
ax.set_title('Stage 7 — Defection Layer Distribution')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/stage7/defection_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Per-confusion-pair breakdown ----
# Count each (true, pred) pair and their median defection layer
pair_stats = defaultdict(list)
for true_cls, pred_cls, dl in defections:
    if dl is not None:
        pair_stats[(true_cls, pred_cls)].append(dl)

# Sort by count descending, take top 12
top_pairs = sorted(pair_stats.items(), key=lambda kv: -len(kv[1]))[:12]

pair_labels  = [f"{CIFAR10_CLASSES[t][:4]}→{CIFAR10_CLASSES[p][:4]}" for (t, p), _ in top_pairs]
pair_counts  = [len(v) for _, v in top_pairs]
pair_medians = [float(np.median(v)) for _, v in top_pairs]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Stage 7 — Top Confusion Pairs: Count and Defection Layer', fontsize=12)

axes[0].barh(pair_labels, pair_counts, color='steelblue')
axes[0].set_xlabel('Number of misclassified images')
axes[0].set_title('Confusion pair frequency')
axes[0].invert_yaxis()
axes[0].grid(True, axis='x', alpha=0.3)

colors = cm.RdYlGn_r(np.array(pair_medians) / (L - 1))
axes[1].barh(pair_labels, pair_medians, color=colors)
axes[1].set_xlabel('Median defection layer (0=L1, 7=L8)')
axes[1].set_title('When does each pair defect?  (green=late, red=early)')
axes[1].set_xlim(-0.5, L - 0.5)
axes[1].set_xticks(range(L))
axes[1].set_xticklabels(LAYER_LABELS)
axes[1].invert_yaxis()
axes[1].grid(True, axis='x', alpha=0.3)

fig.tight_layout()
fig.savefig('experiments/stage7/defection_by_pair.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"{'Pair':<22} {'Count':>6} {'Median defect layer':>20} {'Interpretation'}")
print('-' * 70)
for (t, p), vals in top_pairs:
    med = np.median(vals)
    interp = 'early (never right)' if med <= 2 else ('late (collapse)' if med >= 5 else 'mid')
    print(f"{CIFAR10_CLASSES[t]:<11}→{CIFAR10_CLASSES[p]:<10} {len(vals):>6} "
          f"  {LAYER_LABELS[int(med)]:>8} ({med:.1f})     {interp}")

---

## 4. Per-Image Trajectory Heatmaps

For selected misclassified images, plot the full 10-class × 8-layer cosine similarity heatmap.
Each cell shows how similar the image's activation at that layer is to that class's centroid.

**What to look for:**
- The true class row should start high (bright) and fade
- The predicted (wrong) class row should start low and rise — crossing the true class at the defection layer
- The visual payoff: watching the model commit to the wrong class as depth increases

Two groups are shown:
- **Early defectors** (defection layer ≤ 2) — the model was confused from the start
- **Late defectors** (defection layer ≥ 5) — the model had it right until the collapse region

In [ ]:
# Build defection map: global index → defection layer (int or None)
defection_map = {}
misclassified_list = misclassified_idx.tolist()
for idx, (_, _, dl) in zip(misclassified_list, defections):
    defection_map[idx] = dl

# Select early and late defectors — guard against stored None values
early = [idx for idx in misclassified_list
         if defection_map[idx] is not None and defection_map[idx] <= 2][:6]
late  = [idx for idx in misclassified_list
         if defection_map[idx] is not None and defection_map[idx] >= 5][:6]

print(f"Early defectors (layer ≤ L3): {len(early)} selected")
print(f"Late defectors  (layer ≥ L6): {len(late)}  selected")

In [ ]:
# Load images for display alongside heatmaps
# We need x_all — collect it now (reuses cached backbone pass implicitly)
print("Collecting images for heatmap display...")
z_all, logits_full, x_all, labels_check = analyzer.collect_all(max_samples=10000)
assert (labels_check == labels_all).all(), "Label mismatch — re-run collect_trajectories"
print("Ready.")

In [ ]:
def plot_trajectory_heatmap_with_imgs(
    indices, title, save_path, x_all, trajs, labels_all, preds_all,
    layer_cents, defection_map, n_show=6
):
    import matplotlib.gridspec as gridspec

    n = min(n_show, len(indices))
    if n == 0:
        print(f"  No samples for: {title}"); return

    fig = plt.figure(figsize=(n * 2.8, 6.0))
    fig.suptitle(title, fontsize=12, y=1.01)
    gs  = gridspec.GridSpec(2, n, height_ratios=[1, 2.5], hspace=0.1, wspace=0.25)

    last_im = None
    for col, idx in enumerate(indices[:n]):
        true_cls = labels_all[idx].item()
        pred_cls = preds_all[idx].item()
        dl       = defection_map.get(idx)

        # Image
        ax_img = fig.add_subplot(gs[0, col])
        ax_img.imshow(denormalize(x_all[idx]).permute(1, 2, 0).numpy())
        ax_img.axis('off')
        ax_img.set_title(
            f"\u2192 {CIFAR10_CLASSES[true_cls][:5]}\n"
            f"pred: {CIFAR10_CLASSES[pred_cls][:5]}\n"
            f"defect: {'L'+str(dl+1) if dl is not None else 'none'}",
            fontsize=7.5,
        )

        # Heatmap
        sims = np.zeros((10, L))
        for l in range(L):
            sims[:, l] = analyzer.layer_class_similarities(
                trajs[l][idx], layer_cents[l]
            ).numpy()

        ax_hm = fig.add_subplot(gs[1, col])
        last_im = ax_hm.imshow(sims, aspect='auto', cmap='RdYlGn', vmin=-0.1, vmax=0.6)
        ax_hm.set_xticks(range(L))
        ax_hm.set_xticklabels(LAYER_LABELS, fontsize=7)
        ax_hm.set_xlabel('Layer', fontsize=8)

        if col == 0:
            ax_hm.set_yticks(range(10))
            ax_hm.set_yticklabels(
                [CIFAR10_CLASSES[c][:5] for c in range(10)], fontsize=7
            )
        else:
            ax_hm.set_yticks([])

        # Blue box = true class row, red box = pred class row
        for spine in ax_hm.spines.values(): spine.set_visible(False)
        ax_hm.add_patch(plt.Rectangle(
            (-0.5, true_cls - 0.5), L, 1,
            fill=False, edgecolor='steelblue', lw=2, clip_on=False))
        ax_hm.add_patch(plt.Rectangle(
            (-0.5, pred_cls - 0.5), L, 1,
            fill=False, edgecolor='tomato', lw=2, clip_on=False))

        # White dashed line at defection layer
        if dl is not None:
            ax_hm.axvline(dl, color='white', lw=1.5, ls='--', alpha=0.9)

    if last_im is not None:
        fig.colorbar(last_im, ax=fig.axes[-L:], shrink=0.7,
                     label='cos-sim to class centroid')
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_trajectory_heatmap_with_imgs(
    early, 'Early Defectors (defect at L1–L3): model confused from the start',
    'experiments/stage7/heatmap_early.png',
    x_all, trajs, labels_all, preds_all, layer_cents, defection_map,
)

plot_trajectory_heatmap_with_imgs(
    late, 'Late Defectors (defect at L6–L8): model loses signal at collapse layer',
    'experiments/stage7/heatmap_late.png',
    x_all, trajs, labels_all, preds_all, layer_cents, defection_map,
)

---

## 5. Confusion-Pair Trajectory Profiles

For the top confusion pairs, plot the average per-layer distance to *true class* vs *predicted class* centroid.
The crossing point — where the predicted class distance drops below the true class distance — is the average defection layer for that pair.

This shows whether different confusion types have structurally different error modes.

In [ ]:
top5_pairs = [(t, p) for (t, p), _ in top_pairs[:5]]

fig, axes = plt.subplots(1, len(top5_pairs), figsize=(len(top5_pairs) * 3.2, 3.5),
                          sharey=True)
fig.suptitle('Stage 7 — Per-Pair Trajectory: true-class vs predicted-class distance',
              fontsize=11)

for ax, (t, p) in zip(axes, top5_pairs):
    true_dists = []   # distance to true class
    pred_dists = []   # distance to predicted class

    for idx, (tc, pc, dl) in zip(misclassified_list, defections):
        if tc != t or pc != p: continue
        traj_i = [trajs[l][idx] for l in range(L)]
        true_d = analyzer.trajectory_divergence_curve(traj_i, t, layer_cents)
        pred_d = analyzer.trajectory_divergence_curve(traj_i, p, layer_cents)
        true_dists.append(true_d)
        pred_dists.append(pred_d)

    if not true_dists:
        ax.set_title(f"{CIFAR10_CLASSES[t][:4]}→{CIFAR10_CLASSES[p][:4]}\n(no data)")
        continue

    true_mean = torch.stack(true_dists).mean(0).numpy()
    pred_mean = torch.stack(pred_dists).mean(0).numpy()

    ax.plot(range(L), true_mean, 'o-', color='steelblue',
            label=CIFAR10_CLASSES[t][:5], lw=2)
    ax.plot(range(L), pred_mean, 's--', color='tomato',
            label=CIFAR10_CLASSES[p][:5], lw=2)
    ax.set_xticks(range(L))
    ax.set_xticklabels(LAYER_LABELS, fontsize=7)
    ax.set_title(f"{CIFAR10_CLASSES[t][:4]}→{CIFAR10_CLASSES[p][:4]}  (n={len(true_dists)})",
                  fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Mean cosine distance to centroid')
fig.tight_layout()
fig.savefig('experiments/stage7/pair_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Summary

Fill in after running:

| Finding | Result | Interpretation |
|---------|--------|-----------------|
| Collapse layer (Section 1) | ___ | Lowest separation ratio — confirms Stage 4 |
| Largest correct/incorrect gap (Section 2) | ___ | Most diagnostic layer in the network |
| % misclassified with defection found | ___ | Rest were confused at every layer |
| Early defections (≤ L3) | ___ % | Model never had correct representation |
| Late defections (≥ L6) | ___ % | Collapse layer directly caused error |
| Most frequent confusion pair | ___ | |
| Earliest-defecting pair | ___ | Hardest structural confusion |
| Latest-defecting pair | ___ | Collapse-induced confusion — fixable? |

**Why this is novel:**
Standard tools (GradCAM, probing, saliency maps) answer *where in the image* or *what class*.
Trajectory divergence answers *at which layer* — depth-resolved, per-sample, with no additional training.
The defection layer is a new quantity: the computational moment when the model's internal representation commits to the wrong answer.
The split between early defectors (wrong from the start) and late defectors (wrong only at the collapse layer) has direct implications for where targeted fine-tuning or architectural changes would help most.